## SExtractor detection
- in preparation for construction empirical PSFs using PSFeX

In [4]:
%matplotlib inline
import os
import glob
import numpy as np
from galfitx.source_detection import SExtractor_HDR
from astropy.io import fits
import astropy.units as u
import matplotlib.pyplot as plt
import shutil
from astropy.stats import sigma_clipped_stats
from astropy.table import Table
from astropy.io import ascii
from astropy.visualization import simple_norm
from pathlib import Path
from typing import Optional, Tuple, List
import os
import sys   

In [5]:
sci_dir = '/RS2423/JWST/grism/data/direct_image/'
sci_files = sorted(glob.glob(os.path.join(sci_dir, '*resample.fits')))
fwhm_arcsec_list = {'F115W': 0.06, 'F200W':0.073, 'F356W':0.14} #estimated fwhm
print(sci_files)
print(len(sci_files))

['/RS2423/JWST/grism/data/direct_image/nircam_F115W_mosaic_resample.fits', '/RS2423/JWST/grism/data/direct_image/nircam_F200W_mosaic_resample.fits', '/RS2423/JWST/grism/data/direct_image/nircam_F356W_mosaic_resample.fits']
3


In [6]:
for i, sci_file in enumerate(sci_files):
    band = sci_file.split('/')[-1].split('_')[1]
    print(f"Processing file {i+1}/{len(sci_files)}/: {sci_file}, {band}")
    out_path = "/RS2423/JWST/grism/data/direct_image/sextractor/"+band+"/"
    os.makedirs(out_path, exist_ok=True)
    # Open the FITS file
    with fits.open(sci_file) as hdul:
        data = hdul[1].data
        header = hdul[1].header
    coverage_mask = ((data == 0) | np.isnan(data))  # True for pixels with zero coverage or NaN
    print(f"Shape: {data.shape}; coverage mask: {np.sum(coverage_mask)} pixels")
    pixel_sr = header['PIXAR_SR']
    pixel_scale = np.sqrt(header['PIXAR_A2']) #arcsec/pixel
    mag_zeropoint = -2.5 * np.log10((u.MJy / u.sr * (pixel_sr*u.sr**2) / (3631 * u.Jy)).cgs.value)
    print(f"zeropoint = {mag_zeropoint:.3f}, pixel_scale = {pixel_scale:.3f}")
    outtab, outsegm = SExtractor_HDR(
        filename=sci_file,
        file_ext=1,
        catalog_name=("coldcat_"+band, "hotcat_"+band, "outcat_"+band),
        segmap_name=("coldseg_"+band+".fits", "hotseg_"+band+".fits", "outseg_"+band+".fits"),
        path=out_path,
        back_type=(True, True),
        back_value=(0.0, 0.0),
        detect_thresh=(3, 2),
        back_size=(128, 32),
        mag_zeropoint=mag_zeropoint,
        pixel_scale=pixel_scale,
        fwhm_arcsec=fwhm_arcsec_list[band],
        verbose=True,
        coverage_mask = coverage_mask,
        nnw_sex = '/home/zhanghan/sextractor/config/default.nnw'
    )

Processing file 1/3/: /RS2423/JWST/grism/data/direct_image/nircam_F115W_mosaic_resample.fits, F115W
Shape: (12000, 21000); coverage mask: 31152571 pixels
zeropoint = 28.967, pixel_scale = 0.020
**********Cold Detection**********

Source Extraction Begins.......
detection image: /RS2423/JWST/grism/data/direct_image/nircam_F115W_mosaic_resample.fits


Weight Image Preprocessing Finished
  weight_type = NONE


Automatic Background Estimation and Subtraction Finished
  background size = 128
  background filter size = 3


Detection Finished
  detect_minarea = 5
  detect_thresh = 3
  threshold (above background) = 0.036176733672618866
Found 25970 sources.


Deblending skipped.


Cleaning skipped.


There are 25970 objects in the catalog.

obj_crowded [False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,